# Object Detection — Hard Hat Detection
## Full Pipeline: Data Preparation → Training → Evaluation → Deployment

**Dataset path on Kaggle:** `/kaggle/input/hard-hat-detection/`

Pipeline steps:
1. Install Dependencies & Import Libraries
2. Explore & Visualize Dataset
3. Convert Pascal VOC XML → YOLO Format
4. Train YOLOv8 Model
5. Evaluate Model (mAP, Precision, Recall)
6. Run Inference on Test Images
7. Deploy with Gradio

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics gradio

## 2. Import Libraries

In [ ]:
import os
import glob
import shutil
import random
import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET

from collections import Counter
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

print("All libraries imported successfully")

## 3. Define Dataset Paths

In [ ]:
# Dataset is available directly on Kaggle — no upload needed
images_dir      = "/kaggle/input/datasets/andrewmvd/hard-hat-detection/images"
annotations_dir = "/kaggle/input/datasets/andrewmvd/hard-hat-detection/annotations"

print("Images folder exists:     ", os.path.exists(images_dir))
print("Annotations folder exists:", os.path.exists(annotations_dir))

## 4. Count Images and XML Annotations

In [ ]:
image_extensions = (".jpg", ".jpeg", ".png")

image_files = [
    f for f in os.listdir(images_dir)
    if f.lower().endswith(image_extensions)
]

annotation_files = [
    f for f in os.listdir(annotations_dir)
    if f.lower().endswith(".xml")
]

print("Number of images:           ", len(image_files))
print("Number of XML annotations:  ", len(annotation_files))

## 5. Read Classes and Object Counts

In [ ]:
class_counter = Counter()

for xml_file in annotation_files:
    xml_path = os.path.join(annotations_dir, xml_file)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip()
        class_counter[class_name] += 1

print("Classes and object counts:")
for class_name, count in class_counter.items():
    print(f"  {class_name}: {count}")

## 6. Show Class Distribution

In [ ]:
class_distribution_df = pd.DataFrame(
    list(class_counter.items()),
    columns=["Class", "Object Count"]
)

print(class_distribution_df)

plt.figure(figsize=(8, 5))
plt.bar(class_distribution_df["Class"], class_distribution_df["Object Count"], color=["#4CAF50","#F44336","#2196F3"])
plt.title("Object Count per Class")
plt.xlabel("Class")
plt.ylabel("Number of Objects")
plt.tight_layout()
plt.show()

## 7. Visualize Original XML Annotations

In [ ]:
def draw_boxes_from_xml(image_path, xml_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not read image: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for obj in root.findall("object"):
        class_name = obj.find("name").text.strip()
        bbox = obj.find("bndbox")
        xmin = int(float(bbox.find("xmin").text))
        ymin = int(float(bbox.find("ymin").text))
        xmax = int(float(bbox.find("xmax").text))
        ymax = int(float(bbox.find("ymax").text))
        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (255, 0, 0), 2)
        cv2.putText(image, class_name, (xmin, max(ymin - 5, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 0, 0), 1)
    return image

sample_files = random.sample(image_files, 6)
plt.figure(figsize=(16, 9))

for i, image_file in enumerate(sample_files):
    image_name = os.path.splitext(image_file)[0]
    image_path = os.path.join(images_dir, image_file)
    xml_path   = os.path.join(annotations_dir, image_name + ".xml")
    image = draw_boxes_from_xml(image_path, xml_path)
    plt.subplot(2, 3, i + 1)
    plt.imshow(image)
    plt.axis("off")
    plt.title(image_file, fontsize=9)

plt.tight_layout()
plt.show()

## 8. Define Final Classes

In [ ]:
class_names = ["helmet", "head", "person"]
class_to_id  = {name: idx for idx, name in enumerate(class_names)}

print("Class mapping:", class_to_id)

## 9. Split Dataset (70 / 20 / 10)

In [ ]:
image_files_sorted = sorted(image_files)

train_files, temp_files = train_test_split(
    image_files_sorted, test_size=0.30, random_state=42, shuffle=True
)
val_files, test_files = train_test_split(
    temp_files, test_size=0.33, random_state=42, shuffle=True
)

print("Train images:     ", len(train_files))
print("Validation images:", len(val_files))
print("Test images:      ", len(test_files))

## 10. Create YOLO Folder Structure

In [ ]:
yolo_base = "/kaggle/working/hard_hat_yolo"

if os.path.exists(yolo_base):
    shutil.rmtree(yolo_base)

for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(yolo_base, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(yolo_base, split, "labels"), exist_ok=True)

print("YOLO folder structure created at:", yolo_base)

## 11. Convert Bounding Boxes to YOLO Format

In [ ]:
def convert_bbox_to_yolo(image_width, image_height, bbox):
    xmin, ymin, xmax, ymax = bbox
    xmin = max(0, min(xmin, image_width))
    xmax = max(0, min(xmax, image_width))
    ymin = max(0, min(ymin, image_height))
    ymax = max(0, min(ymax, image_height))
    box_width  = xmax - xmin
    box_height = ymax - ymin
    x_center   = (xmin + box_width  / 2) / image_width
    y_center   = (ymin + box_height / 2) / image_height
    box_width  /= image_width
    box_height /= image_height
    return x_center, y_center, box_width, box_height

## 12. Convert XML Files to YOLO TXT Labels

In [ ]:
def convert_and_copy(files_list, split):
    skipped = []
    empty   = []

    for image_file in files_list:
        image_name   = os.path.splitext(image_file)[0]
        src_img      = os.path.join(images_dir, image_file)
        src_xml      = os.path.join(annotations_dir, image_name + ".xml")
        dst_img      = os.path.join(yolo_base, split, "images", image_file)
        dst_label    = os.path.join(yolo_base, split, "labels", image_name + ".txt")

        if not os.path.exists(src_xml):
            skipped.append(image_file)
            continue

        shutil.copy(src_img, dst_img)

        tree   = ET.parse(src_xml)
        root   = tree.getroot()
        width  = int(root.find("size").find("width").text)
        height = int(root.find("size").find("height").text)
        lines  = []

        for obj in root.findall("object"):
            class_name = obj.find("name").text.strip()
            if class_name not in class_to_id:
                continue
            bbox = obj.find("bndbox")
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
            xc, yc, bw, bh = convert_bbox_to_yolo(width, height, (xmin, ymin, xmax, ymax))
            if bw <= 0 or bh <= 0:
                continue
            lines.append(f"{class_to_id[class_name]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

        if not lines:
            empty.append(image_file)

        with open(dst_label, "w") as f:
            f.write("\n".join(lines))

    return skipped, empty


skipped_train, empty_train = convert_and_copy(train_files, "train")
skipped_val,   empty_val   = convert_and_copy(val_files,   "val")
skipped_test,  empty_test  = convert_and_copy(test_files,  "test")

print("Conversion completed")
for split, sk, em in [("train", skipped_train, empty_train),
                       ("val",   skipped_val,   empty_val),
                       ("test",  skipped_test,  empty_test)]:
    imgs = len(os.listdir(os.path.join(yolo_base, split, "images")))
    print(f"  {split:5s} → {imgs} images | skipped: {len(sk)} | empty labels: {len(em)}")

## 13. Visualize Converted YOLO Labels

In [ ]:
def draw_yolo_boxes(image_path, label_path, class_names):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not read: {image_path}")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w, _ = image.shape
    if os.path.exists(label_path):
        for line in open(label_path).readlines():
            vals = line.strip().split()
            if len(vals) != 5:
                continue
            cls_id = int(vals[0])
            xc, yc, bw, bh = map(float, vals[1:])
            x1 = int((xc - bw / 2) * w)
            y1 = int((yc - bh / 2) * h)
            x2 = int((xc + bw / 2) * w)
            y2 = int((yc + bh / 2) * h)
            cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(image, class_names[cls_id], (x1, max(y1 - 5, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1)
    return image

sample_train = random.sample(os.listdir(os.path.join(yolo_base, "train", "images")), 6)
plt.figure(figsize=(16, 9))

for i, image_file in enumerate(sample_train):
    image_name = os.path.splitext(image_file)[0]
    img_path   = os.path.join(yolo_base, "train", "images", image_file)
    lbl_path   = os.path.join(yolo_base, "train", "labels", image_name + ".txt")
    img = draw_yolo_boxes(img_path, lbl_path, class_names)
    plt.subplot(2, 3, i + 1)
    plt.imshow(img)
    plt.axis("off")
    plt.title(image_file, fontsize=9)

plt.tight_layout()
plt.show()

## 14. Create data.yaml

In [ ]:
data_yaml = {
    "path"  : yolo_base,
    "train" : "train/images",
    "val"   : "val/images",
    "test"  : "test/images",
    "nc"    : len(class_names),
    "names" : class_names
}

data_yaml_path = os.path.join(yolo_base, "data.yaml")

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("data.yaml created:")
with open(data_yaml_path, "r") as f:
    print(f.read())

## 15. Load YOLOv8 Pretrained Model

We use **YOLOv8n** (nano) — the lightest version, suitable for Kaggle free GPU.

### Why YOLOv8?
- State-of-the-art real-time object detection
- Anchor-free detection head for better accuracy
- Excellent trade-off between speed and accuracy
- Easy transfer learning from pretrained COCO weights

In [ ]:
model = YOLO("yolov8s.pt")

print("Model loaded successfully")
model.info()

## 16. Train YOLOv8 on Hard Hat Dataset

In [ ]:
results = model.train(
    data=data_yaml_path,
    epochs=50,
    imgsz=640,
    batch=16,               
    name="hard_hat_yolov8_v2",
    project="/kaggle/working/runs",
    patience=15,
    cls=1.5,
    box=9.0,
    fliplr=0.5,
    flipud=0.1,
    mosaic=1.0,
    mixup=0.2,
    copy_paste=0.3,
    save=True,
    plots=True,
    verbose=True
)

print("Training completed!")
print("Results saved to:", results.save_dir)

## 17. Show Training Results (Loss & Metrics Curves)

In [ ]:
train_dir       = "/kaggle/working/runs/hard_hat_yolov8_v2"
best_model_path = "/kaggle/working/runs/hard_hat_yolov8_v2/weights/best.pt"

results_df = pd.read_csv(os.path.join(train_dir, "results.csv"))
results_df.columns = results_df.columns.str.strip()

print(results_df.tail(5))

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle("YOLOv8 Training Results", fontsize=16, fontweight="bold")

ep = results_df["epoch"]

axes[0, 0].plot(ep, results_df["train/box_loss"], color="blue",   label="Box Loss")
axes[0, 0].set_title("Train Box Loss"); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

axes[0, 1].plot(ep, results_df["train/cls_loss"], color="orange", label="Class Loss")
axes[0, 1].set_title("Train Classification Loss"); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)

axes[0, 2].plot(ep, results_df["train/dfl_loss"], color="green",  label="DFL Loss")
axes[0, 2].set_title("Train DFL Loss"); axes[0, 2].legend(); axes[0, 2].grid(alpha=0.3)

axes[1, 0].plot(ep, results_df["metrics/precision(B)"], color="red",    label="Precision")
axes[1, 0].plot(ep, results_df["metrics/recall(B)"],    color="purple",  label="Recall")
axes[1, 0].set_title("Precision & Recall"); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

axes[1, 1].plot(ep, results_df["metrics/mAP50(B)"],    color="darkblue", label="mAP@50")
axes[1, 1].plot(ep, results_df["metrics/mAP50-95(B)"], color="cyan",     label="mAP@50-95")
axes[1, 1].set_title("mAP Scores"); axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

axes[1, 2].plot(ep, results_df["val/box_loss"], color="brown", label="Val Box Loss")
axes[1, 2].plot(ep, results_df["val/cls_loss"], color="gray",  label="Val Cls Loss")
axes[1, 2].set_title("Validation Losses"); axes[1, 2].legend(); axes[1, 2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/kaggle/working/training_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved")

## 18. Evaluate Model on Test Set

**Metrics explained:**
- **mAP@50**: Mean Average Precision at IoU threshold 0.50
- **mAP@50-95**: Mean Average Precision averaged over IoU thresholds 0.50 → 0.95
- **Precision**: Of all predicted boxes, how many were correct
- **Recall**: Of all actual objects, how many were detected

In [ ]:
best_model_path = "/kaggle/working/runs/hard_hat_yolov8_v2/weights/best.pt"
trained_model   = YOLO(best_model_path)

print("Best model loaded from:", best_model_path)

eval_results = trained_model.val(
    data=data_yaml_path,
    split="test",
    imgsz=640,
    conf=0.25,
    iou=0.5,
    verbose=True
)

print("\n" + "=" * 50)
print("TEST SET EVALUATION RESULTS")
print("=" * 50)
print(f"mAP@50:    {eval_results.box.map50:.4f}")
print(f"mAP@50-95: {eval_results.box.map:.4f}")
print(f"Precision: {eval_results.box.mp:.4f}")
print(f"Recall:    {eval_results.box.mr:.4f}")
print("=" * 50)

print("\nPer-Class Metrics:")
print(f"{'Class':<12} {'Precision':>12} {'Recall':>10} {'mAP@50':>10}")
print("-" * 50)
for i, cls_name in enumerate(class_names):
    p    = eval_results.box.p[i]    if i < len(eval_results.box.p)    else 0
    r    = eval_results.box.r[i]    if i < len(eval_results.box.r)    else 0
    ap50 = eval_results.box.ap50[i] if i < len(eval_results.box.ap50) else 0
    print(f"{cls_name:<12} {p:>12.4f} {r:>10.4f} {ap50:>10.4f}")

# Bar chart
metrics = {
    "mAP@50": eval_results.box.map50,
    "mAP@50-95": eval_results.box.map,
    "Precision": eval_results.box.mp,
    "Recall": eval_results.box.mr
}

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(metrics.keys(), metrics.values(),
              color=["#2196F3","#4CAF50","#FF9800","#E91E63"],
              width=0.5, edgecolor="white", linewidth=1.5)

for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
            f"{val:.4f}", ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(0, 1.1)
ax.set_title("YOLOv8 — Test Set Evaluation Metrics", fontsize=14, fontweight="bold")
ax.set_ylabel("Score", fontsize=12)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/evaluation_metrics.png", dpi=150, bbox_inches="tight")
plt.show()

## 19. Visualize Confusion Matrix

In [ ]:
confusion_matrix_path = "/kaggle/working/runs/hard_hat_yolov8_v2/confusion_matrix.png"

if os.path.exists(confusion_matrix_path):
    img = plt.imread(confusion_matrix_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis("off")
    plt.title("Confusion Matrix", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Confusion matrix not found. Available files:")
    print(os.listdir("/kaggle/working/runs/hard_hat_yolov8"))

## 20. Run Inference on Test Images

In [ ]:
test_images_dir  = os.path.join(yolo_base, "test", "images")
test_image_files = os.listdir(test_images_dir)
sample_images    = random.sample(test_image_files, min(6, len(test_image_files)))

CLASS_COLORS = {0: (0, 200, 0), 1: (200, 0, 0), 2: (0, 0, 200)}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("YOLOv8 Inference on Test Images", fontsize=16, fontweight="bold")

for idx, image_file in enumerate(sample_images):
    image_path = os.path.join(test_images_dir, image_file)

    result = trained_model.predict(source=image_path, conf=0.25, iou=0.5, verbose=False)[0]

    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        cls_id = int(box.cls[0])
        conf   = float(box.conf[0])
        color  = CLASS_COLORS.get(cls_id, (128, 128, 128))
        label  = f"{class_names[cls_id]} {conf:.2f}"
        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)
        cv2.rectangle(image, (x1, y1 - 18), (x1 + len(label) * 8, y1), color, -1)
        cv2.putText(image, label, (x1 + 2, y1 - 4),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1)

    row, col = idx // 3, idx % 3
    axes[row, col].imshow(image)
    axes[row, col].axis("off")
    axes[row, col].set_title(f"{image_file} — {len(result.boxes)} detections", fontsize=9)

plt.tight_layout()
plt.savefig("/kaggle/working/inference_results.png", dpi=150, bbox_inches="tight")
plt.show()
print("Inference results saved")

## Summary

| Step | Description |
|------|-------------|
| Dataset | Loaded directly from `/kaggle/input/hard-hat-detection/` |
| Preprocessing | Pascal VOC XML → YOLO TXT format |
| Split | 70% train / 20% val / 10% test |
| Model | YOLOv8n pretrained on COCO |
| Training | 30 epochs, 640px, batch=16, early stopping patience=10 |
| Evaluation | mAP@50, mAP@50-95, Precision, Recall on test set |
| Inference | Visual results on 6 random test images |
| Deployment | Gradio web app with live inference |